# Active Learning: BALD, Core-Set, and BADGE

This notebook explores uncertainty-based and diversity-based sample acquisition strategies for active learning:
1. **BALD**: Information-theoretic Bayesian Active Learning by Disagreement isolating epistemic uncertainty.
2. **Core-Set**: Geometric $k$-Center Greedy coverage on latent representations.
3. **BADGE**: Hybrid active learning via diverse gradient embeddings.

**Decision Boundary (When NOT to Use)**:
- Do not use raw predictive entropy on noisy or uncurated datasets, as it wastes labeling budget on irreducible aleatoric label noise rather than epistemic parameter uncertainty (use BALD).
- Do not use Core-Set when data contains extreme geometric outliers, as $k$-Center Greedy prioritizes distant meaningless points.

**Topic Bridges & Connects To**:
- `06_federated_learning_fedavg_fedprox_scaffold_and_secagg` (Decentralized client data selection)
- `02/01_value_based_rl_dqn_rainbow_and_qr_dqn` (Active exploration versus exploitation)

## 2. Mathematical Formulations & Theoretical Foundations

- **BALD Epistemic Mutual Information**:
  $$\mathbb{I}(y; \omega \mid x) = \mathbb{H}(\bar{p}) - \frac{1}{M} \sum_{m=1}^M \mathbb{H}(p^{(m)}), \quad \mathcal{O}(M \cdot C)$$
- **Core-Set k-Center Greedy Coverage**:
  $$u^* = \arg\max_{u \in \mathcal{U}} \min_{l \in \mathcal{L}} \|\phi(u) - \phi(l)\|_2, \quad \mathcal{O}(N_U \cdot N_L \cdot D)$$
- **BADGE Gradient Embedding**:
  $$g_x = (p - e_{\hat{y}}) \otimes z(x) \in \mathbb{R}^{C \times D}, \quad \mathcal{O}(C \cdot D)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "07_active_learning_bald_coreset_and_badge.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. BALD Epistemic Mutual Information Score
def bald_score(mc_probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    mean_probs = mc_probs.mean(dim=0)
    entropy_mean = -(mean_probs * torch.log(mean_probs + eps)).sum(dim=-1)
    mean_entropy = -(mc_probs * torch.log(mc_probs + eps)).sum(dim=-1).mean(dim=0)
    return entropy_mean - mean_entropy

bald_calc = bald_score(golden["mc_probs"])
diff_bald = torch.max(torch.abs(bald_calc - golden["bald_score_ref"])).item()
print(f"BALD score max abs error: {diff_bald:.2e}")
assert diff_bald < 1e-5

In [ ]:
# 2. Core-Set k-Center Greedy Minimum Distances
def coreset_min_distances(unlabeled_feats: torch.Tensor, labeled_feats: torch.Tensor) -> torch.Tensor:
    dist_matrix = torch.cdist(unlabeled_feats, labeled_feats, p=2)
    min_dists, _ = dist_matrix.min(dim=1)
    return min_dists

min_dists_calc = coreset_min_distances(golden["unlabeled_feats"], golden["labeled_feats"])
diff_core = torch.max(torch.abs(min_dists_calc - golden["coreset_min_dists_ref"])).item()
print(f"Core-Set min dists max abs error: {diff_core:.2e}")
assert diff_core < 1e-5

best_idx_calc = min_dists_calc.argmax().item()
assert best_idx_calc == golden["best_unlabeled_idx_ref"].item()

In [ ]:
# 3. BADGE Gradient Embeddings
def badge_gradient_embeddings(feats: torch.Tensor, probs: torch.Tensor) -> torch.Tensor:
    pred_class = probs.argmax(dim=-1)
    c = probs.shape[-1]
    one_hot = F.one_hot(pred_class, num_classes=c).float()
    prob_diff = probs - one_hot
    return (prob_diff.unsqueeze(2) * feats.unsqueeze(1)).flatten(start_dim=1)

badge_calc = badge_gradient_embeddings(golden["feats"], golden["probs"])
diff_badge = torch.max(torch.abs(badge_calc - golden["badge_embeds_ref"])).item()
print(f"BADGE Embeddings max abs error: {diff_badge:.2e}")
assert diff_badge < 1e-5

In [ ]:
# 4. Anti-Pattern / Misuse Failure Demonstration: Aleatoric Noise Trap in Raw Entropy
# Point A: Pure aleatoric noise (50/50 coin flip, all models agree it's random)
# Point B: High epistemic uncertainty (model 1 says class 0, model 2 says class 1)
mc_aleatoric = torch.tensor([[[0.5, 0.5]], [[0.5, 0.5]]]) # M=2, B=1, C=2
mc_epistemic = torch.tensor([[[0.99, 0.01]], [[0.01, 0.99]]])

# Raw entropy treats both as identical high entropy:
mean_aleatoric = mc_aleatoric.mean(dim=0)
mean_epistemic = mc_epistemic.mean(dim=0)
h_aleatoric = -(mean_aleatoric * torch.log(mean_aleatoric)).sum(dim=-1).item()
h_epistemic = -(mean_epistemic * torch.log(mean_epistemic)).sum(dim=-1).item()
assert abs(h_aleatoric - h_epistemic) < 1e-3, "Raw entropy cannot distinguish epistemic from aleatoric noise"

# BALD correctly identifies that Point A has zero epistemic value:
bald_aleatoric = bald_score(mc_aleatoric).item()
bald_epistemic = bald_score(mc_epistemic).item()
assert bald_aleatoric < 1e-5, "Anti-Pattern: Aleatoric noise must produce zero BALD score"
assert bald_epistemic > 0.6, "BALD Failure: Disagreeing models must produce high epistemic score"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'BALD Epistemic Score':<32} | {diff_bald:<15.2e} | PASSED")
print(f"{'Core-Set Min Distances':<32} | {diff_core:<15.2e} | PASSED")
print(f"{'BADGE Gradient Embeddings':<32} | {diff_badge:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **BALD**: Distinguishes epistemic parameter disagreement from aleatoric inherent noise, preventing wasted labeling budget.
- [x] **Core-Set**: Formalizes active learning as geometry coverage; $k$-Center Greedy provides a guaranteed 2-approximation.
- [x] **BADGE**: Unifies uncertainty and diversity through output-layer gradient embeddings and $k$-means++ seeding.
- [x] **Noise Robustness**: Epistemic scoring prevents querying unlearnable noisy boundaries.

**Cross-Domain Generalisation**:
Active learning principles scale from visual annotation to drug discovery (actively selecting which molecular compounds to synthesize and test in vitro), materials science (selecting alloy compositions for expensive physical stress testing), and autonomous vehicles (detecting rare edge-case driving maneuvers).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).